In [2]:
# Cell 1. Bulls and Cows MEM Solver

import math
from itertools import permutations
from collections import Counter
from statistics import mean

import numpy as np


def generate_numbers(n: int, digits: str = "0123456789") -> list[str]:
    """
    Generate all n-digit Bulls and Cows candidates without repeated digits.

    By default, leading zero is allowed because the object is treated as
    an n-symbol code rather than an ordinary decimal number.
    """
    if not 1 <= n <= len(digits):
        raise ValueError("n must be between 1 and the number of available digits.")

    return ["".join(p) for p in permutations(digits, n)]


def get_bulls_and_cows_pattern(answer: str, guess: str) -> tuple[int, int]:
    """
    Return Bulls and Cows feedback.

    bulls = correct digit in the correct position
    cows = correct digit in the wrong position

    Since candidates have no repeated digits, this is straightforward.
    """
    bulls = sum(a == g for a, g in zip(answer, guess))
    common = len(set(answer) & set(guess))
    cows = common - bulls
    return bulls, cows


def encode_pattern(pattern: tuple[int, int], n: int) -> int:
    """
    Encode (bulls, cows) as an integer pattern ID.
    """
    bulls, cows = pattern
    return bulls * (n + 1) + cows


def decode_pattern(pattern_id: int, n: int) -> tuple[int, int]:
    """
    Decode an integer pattern ID back to (bulls, cows).
    """
    return pattern_id // (n + 1), pattern_id % (n + 1)


def build_pattern_table(answers: list[str], guesses: list[str], n: int) -> np.ndarray:
    """
    Build pattern_table[answer_index, guess_index] = pattern_id.
    """
    pattern_count = (n + 1) * (n + 1)

    dtype = np.uint8 if pattern_count <= 256 else np.uint16
    table = np.empty((len(answers), len(guesses)), dtype=dtype)

    for i, answer in enumerate(answers):
        for j, guess in enumerate(guesses):
            table[i, j] = encode_pattern(
                get_bulls_and_cows_pattern(answer, guess),
                n,
            )

    return table


def modified_score_from_components(
    block_count_score: float,
    uniformity_score: float,
    lam: float,
) -> float:
    """
    Compute Modified Entropy Maximizing score.

    lam = 0.5 corresponds to normalized entropy.
    lam > 0.5 emphasizes partition diversity.
    lam < 0.5 emphasizes partition uniformity.
    """
    if block_count_score <= 0 or uniformity_score <= 0:
        return 0.0

    return (
        block_count_score ** (2 * lam)
        * uniformity_score ** (2 * (1 - lam))
    )


def metrics_from_counts(
    counts: np.ndarray,
    n_candidates: int,
    pattern_count: int,
    lam: float,
) -> dict:
    """
    Compute partition metrics from feedback-pattern counts.
    """
    if not 0 <= lam <= 1:
        raise ValueError("lam must be between 0 and 1.")

    sizes = counts[counts > 0].astype(np.float64)
    k = len(sizes)

    if k == 0:
        raise ValueError("No nonempty partition blocks.")

    max_block = int(sizes.max())
    expected_remaining = float(np.sum(sizes * sizes) / n_candidates)

    if k <= 1:
        entropy = 0.0
        block_count_score = 0.0
        uniformity_score = 0.0
        normalized_entropy = 0.0
        score = 0.0
    else:
        probs = sizes / n_candidates
        entropy = float(-np.sum(probs * np.log2(probs)))

        max_possible_blocks = min(pattern_count, n_candidates)

        block_count_score = math.log2(k) / math.log2(max_possible_blocks)
        uniformity_score = entropy / math.log2(k)
        normalized_entropy = entropy / math.log2(max_possible_blocks)

        score = modified_score_from_components(
            block_count_score=block_count_score,
            uniformity_score=uniformity_score,
            lam=lam,
        )

    return {
        "score": score,
        "lambda": lam,
        "n_blocks": int(k),
        "entropy": entropy,
        "normalized_entropy": normalized_entropy,
        "block_count_score": block_count_score,
        "uniformity_score": uniformity_score,
        "max_block": max_block,
        "expected_remaining": expected_remaining,
    }


def get_partition_counts_from_array(
    pattern_table: np.ndarray,
    candidate_array: np.ndarray,
    guess_index: int,
    pattern_count: int,
) -> np.ndarray:
    """
    Return counts of feedback patterns for a guess over current candidates.
    """
    pattern_ids = pattern_table[candidate_array, guess_index]

    return np.bincount(
        pattern_ids,
        minlength=pattern_count,
    )


def group_candidate_indices_by_pattern(
    pattern_table: np.ndarray,
    candidate_indices: tuple[int, ...],
    candidate_array: np.ndarray,
    guess_index: int,
) -> dict[int, tuple[int, ...]]:
    """
    Split candidate answer indices by feedback pattern.
    """
    pattern_ids = pattern_table[candidate_array, guess_index]

    groups = {}

    for answer_index, pattern_id in zip(candidate_indices, pattern_ids):
        pattern_id = int(pattern_id)

        if pattern_id not in groups:
            groups[pattern_id] = []

        groups[pattern_id].append(answer_index)

    return {
        pattern_id: tuple(indices)
        for pattern_id, indices in groups.items()
    }


def pick_best_guess_modified_entropy(
    answers: list[str],
    guesses: list[str],
    pattern_table: np.ndarray,
    answer_to_index: dict[str, int],
    guess_index_to_answer_index: tuple[int | None, ...],
    candidate_indices: tuple[int, ...],
    guess_indices: tuple[int, ...],
    pattern_count: int,
    lam: float,
) -> dict:
    """
    Pick the best guess using Modified Entropy Maximizing.

    Tie-breaking:
    1. Higher modified score
    2. Lower expected remaining
    3. Lower max block
    4. Prefer guesses that are still possible answers
    5. Lexicographic order
    """
    candidate_array = np.fromiter(
        candidate_indices,
        dtype=np.int32,
        count=len(candidate_indices),
    )

    candidate_set = set(candidate_indices)

    best_guess_index = None
    best_metrics = None
    best_key = None

    for guess_index in guess_indices:
        counts = get_partition_counts_from_array(
            pattern_table=pattern_table,
            candidate_array=candidate_array,
            guess_index=guess_index,
            pattern_count=pattern_count,
        )

        metrics = metrics_from_counts(
            counts=counts,
            n_candidates=len(candidate_indices),
            pattern_count=pattern_count,
            lam=lam,
        )

        answer_index_for_guess = guess_index_to_answer_index[guess_index]

        is_candidate = (
            1
            if answer_index_for_guess is not None
            and answer_index_for_guess in candidate_set
            else 0
        )

        key = (
            metrics["score"],
            -metrics["expected_remaining"],
            -metrics["max_block"],
            is_candidate,
            -guess_index,
        )

        if best_key is None or key > best_key:
            best_key = key
            best_guess_index = guess_index
            best_metrics = metrics

    groups = group_candidate_indices_by_pattern(
        pattern_table=pattern_table,
        candidate_indices=candidate_indices,
        candidate_array=candidate_array,
        guess_index=best_guess_index,
    )

    return {
        "guess_index": best_guess_index,
        "guess": guesses[best_guess_index],
        "groups": groups,
        **best_metrics,
    }


def evaluate_modified_entropy(
    answers: list[str],
    guesses: list[str],
    pattern_table: np.ndarray,
    n: int,
    lam: float = 0.5,
    first_guess: str | None = None,
) -> dict:
    """
    Evaluate MEM on Bulls and Cows.

    Strategy:
    - If first_guess is None, choose the first guess by MEM.
    - After that, recursively choose each next guess by MEM.
    - Return actual solving chains and attempt counts.
    """
    if not 0 <= lam <= 1:
        raise ValueError("lam must be between 0 and 1.")

    pattern_count = (n + 1) * (n + 1)
    correct_pattern_id = encode_pattern((n, 0), n)

    answer_to_index = {
        number: i
        for i, number in enumerate(answers)
    }

    guess_to_index = {
        number: i
        for i, number in enumerate(guesses)
    }

    guess_index_to_answer_index = tuple(
        answer_to_index.get(guess)
        for guess in guesses
    )

    answer_indices = tuple(range(len(answers)))
    all_guess_indices = tuple(range(len(guesses)))

    if first_guess is not None:
        first_guess = first_guess.strip()

        if first_guess not in guess_to_index:
            raise ValueError(f"{first_guess!r} is not in the allowed guess list.")

    memo = {}
    policy = {}

    def solve_state(candidate_indices: tuple[int, ...]) -> dict[int, tuple[int, ...]]:
        candidate_indices = tuple(sorted(candidate_indices))

        if candidate_indices in memo:
            return memo[candidate_indices]

        if len(candidate_indices) == 0:
            raise ValueError("Candidate state is empty.")

        if len(candidate_indices) == 1:
            answer_index = candidate_indices[0]
            final_guess_index = guess_to_index[answers[answer_index]]

            memo[candidate_indices] = {
                answer_index: (final_guess_index,)
            }

            return memo[candidate_indices]

        best = pick_best_guess_modified_entropy(
            answers=answers,
            guesses=guesses,
            pattern_table=pattern_table,
            answer_to_index=answer_to_index,
            guess_index_to_answer_index=guess_index_to_answer_index,
            candidate_indices=candidate_indices,
            guess_indices=all_guess_indices,
            pattern_count=pattern_count,
            lam=lam,
        )

        guess_index = best["guess_index"]
        groups = best["groups"]

        policy[candidate_indices] = best

        chains = {}

        for pattern_id, subgroup in groups.items():
            if pattern_id == correct_pattern_id:
                for answer_index in subgroup:
                    chains[answer_index] = (guess_index,)
                continue

            subgroup_chains = solve_state(subgroup)

            for answer_index, subchain in subgroup_chains.items():
                chains[answer_index] = (guess_index,) + subchain

        memo[candidate_indices] = chains
        return chains

    if first_guess is None:
        chains_by_answer_index = solve_state(answer_indices)
        actual_first_guess = policy[answer_indices]["guess"]
        first_guess_profile = policy[answer_indices]
    else:
        first_guess_index = guess_to_index[first_guess]

        candidate_array = np.fromiter(
            answer_indices,
            dtype=np.int32,
            count=len(answer_indices),
        )

        first_counts = get_partition_counts_from_array(
            pattern_table=pattern_table,
            candidate_array=candidate_array,
            guess_index=first_guess_index,
            pattern_count=pattern_count,
        )

        first_guess_profile = {
            "guess_index": first_guess_index,
            "guess": first_guess,
            **metrics_from_counts(
                counts=first_counts,
                n_candidates=len(answer_indices),
                pattern_count=pattern_count,
                lam=lam,
            ),
        }

        first_groups = group_candidate_indices_by_pattern(
            pattern_table=pattern_table,
            candidate_indices=answer_indices,
            candidate_array=candidate_array,
            guess_index=first_guess_index,
        )

        correct_pattern_id = encode_pattern((n, 0), n)
        chains_by_answer_index = {}

        for pattern_id, subgroup in first_groups.items():
            if pattern_id == correct_pattern_id:
                for answer_index in subgroup:
                    chains_by_answer_index[answer_index] = (first_guess_index,)
                continue

            subgroup_chains = solve_state(subgroup)

            for answer_index, subchain in subgroup_chains.items():
                chains_by_answer_index[answer_index] = (first_guess_index,) + subchain

        actual_first_guess = first_guess

    chains_by_answer = {
        answers[answer_index]: tuple(guesses[guess_index] for guess_index in chain)
        for answer_index, chain in chains_by_answer_index.items()
    }

    attempts_by_answer = {
        answer: len(chain)
        for answer, chain in chains_by_answer.items()
    }

    values = list(attempts_by_answer.values())

    return {
        "algorithm": "Modified Entropy Maximizing for Bulls and Cows",
        "lambda": lam,
        "n": n,
        "first_guess": actual_first_guess,
        "mean": mean(values),
        "max": max(values),
        "results": attempts_by_answer,
        "chains": chains_by_answer,
        "first_guess_profile": first_guess_profile,
        "policy_state_count": len(policy),
        "memo_state_count": len(memo),
    }


def attempt_distribution(result: dict) -> list[dict]:
    """
    Return distribution of actual attempt counts.
    """
    counts = Counter(result["results"].values())
    total = sum(counts.values())

    return [
        {
            "attempts": attempts,
            "count": counts[attempts],
            "probability": counts[attempts] / total,
        }
        for attempts in sorted(counts)
    ]


def get_worst_answers(result: dict) -> list[str]:
    """
    Return answers that required the maximum number of attempts.
    """
    max_attempts = result["max"]

    return sorted(
        answer
        for answer, attempts in result["results"].items()
        if attempts == max_attempts
    )


def format_chain(chain: tuple[str, ...], answer: str) -> str:
    """
    Format a solving chain like 0123-4567-0189-0129(correct).
    """
    guesses = list(chain)

    if len(guesses) == 0:
        return ""

    if guesses[-1] == answer:
        guesses[-1] = guesses[-1] + "(correct)"
    else:
        guesses.append(answer + "(correct)")

    return "-".join(guesses)


def run_bulls_and_cows_mem(n: int) -> list[dict]:
    """
    Run MEM for lambda = 0.0, 0.1, ..., 1.0.

    This is the main function called by the execution cell.
    """
    answers = generate_numbers(n)
    guesses = answers.copy()

    print("Bulls and Cows MEM")
    print("n:", n)
    print("answers:", len(answers))
    print("guesses:", len(guesses))
    print()

    pattern_table = build_pattern_table(answers, guesses, n)

    print("pattern table shape:", pattern_table.shape)
    print("pattern table memory MB:", f"{pattern_table.nbytes / 1024 / 1024:.2f}")
    print()

    lambda_values = [round(i / 10, 1) for i in range(11)]
    lambda_results = []

    for lam in lambda_values:
        result = evaluate_modified_entropy(
            answers=answers,
            guesses=guesses,
            pattern_table=pattern_table,
            n=n,
            lam=lam,
            first_guess=None,
        )

        worst_answers = get_worst_answers(result)

        row = {
            "lambda": lam,
            "first_guess": result["first_guess"],
            "mean": result["mean"],
            "max": result["max"],
            "policy_states": result["policy_state_count"],
            "memo_states": result["memo_state_count"],
            "worst_count": len(worst_answers),
            "worst_answers": worst_answers,
            "result": result,
        }

        lambda_results.append(row)

        print(
            f"lambda={lam:.1f} | "
            f"first={result['first_guess']} | "
            f"mean={result['mean']:.4f} | "
            f"max={result['max']} | "
            f"worst_count={len(worst_answers)}"
        )

    print()
    print("summary:")
    print("lambda,first_guess,mean,max,worst_count")

    for row in lambda_results:
        print(
            f"{row['lambda']:.1f},"
            f"{row['first_guess']},"
            f"{row['mean']:.4f},"
            f"{row['max']},"
            f"{row['worst_count']}"
        )

    best_by_mean = min(lambda_results, key=lambda r: r["mean"])
    best_by_max_then_mean = min(lambda_results, key=lambda r: (r["max"], r["mean"]))

    print()
    print(
        "best by mean:",
        f"lambda={best_by_mean['lambda']:.1f},",
        f"first={best_by_mean['first_guess']},",
        f"mean={best_by_mean['mean']:.4f},",
        f"max={best_by_mean['max']}",
    )

    print(
        "best by max then mean:",
        f"lambda={best_by_max_then_mean['lambda']:.1f},",
        f"first={best_by_max_then_mean['first_guess']},",
        f"mean={best_by_max_then_mean['mean']:.4f},",
        f"max={best_by_max_then_mean['max']}",
    )

    return lambda_results

In [6]:
# Cell 2. Execution cell

N = 3

lambda_results = run_bulls_and_cows_mem(N)

Bulls and Cows MEM
n: 3
answers: 720
guesses: 720

pattern table shape: (720, 720)
pattern table memory MB: 0.49

lambda=0.0 | first=012 | mean=8.2972 | max=11 | worst_count=40
lambda=0.1 | first=012 | mean=5.9222 | max=8 | worst_count=9
lambda=0.2 | first=012 | mean=5.2306 | max=7 | worst_count=13
lambda=0.3 | first=012 | mean=5.1597 | max=7 | worst_count=11
lambda=0.4 | first=012 | mean=5.1236 | max=7 | worst_count=7
lambda=0.5 | first=012 | mean=5.0250 | max=7 | worst_count=2
lambda=0.6 | first=012 | mean=5.0333 | max=7 | worst_count=2
lambda=0.7 | first=012 | mean=5.0333 | max=7 | worst_count=2
lambda=0.8 | first=012 | mean=5.0319 | max=7 | worst_count=2
lambda=0.9 | first=012 | mean=5.0333 | max=7 | worst_count=2
lambda=1.0 | first=012 | mean=5.0319 | max=7 | worst_count=2

summary:
lambda,first_guess,mean,max,worst_count
0.0,012,8.2972,11,40
0.1,012,5.9222,8,9
0.2,012,5.2306,7,13
0.3,012,5.1597,7,11
0.4,012,5.1236,7,7
0.5,012,5.0250,7,2
0.6,012,5.0333,7,2
0.7,012,5.0333,7,2
0.8,0